In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. CONFIGURATION
# ============================================================

storage_account = "stsupplychain2026xx"

bronze_orders_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "bronze/orders/"
)

silver_orders_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "silver/orders/"
)

dq_invalid_orders_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "quality/dq_invalid_records/orders/"
)

dq_metrics_orders_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "quality/dq_metrics/orders/"
)

# ============================================================
# 2. LECTURE DE LA BRONZE
# ============================================================

bronze_orders = (
    spark.read
    .format("delta")
    .load(bronze_orders_path)
)

print(
    "Nombre de lignes Bronze :",
    bronze_orders.count()
)

# ============================================================
# 3. NORMALISATION
# ============================================================

normalized_orders = (
    bronze_orders

    # Nettoyage des identifiants
    .withColumn(
        "order_id",
        F.trim(F.col("order_id"))
    )

    .withColumn(
        "customer_id",
        F.trim(F.col("customer_id"))
    )

    # Normalisation ville / pays
    .withColumn(
        "destination_city",
        F.initcap(
            F.trim(
                F.col("destination_city")
            )
        )
    )

    .withColumn(
        "destination_country",
        F.initcap(
            F.trim(
                F.col("destination_country")
            )
        )
    )

    # Standardisation devise
    .withColumn(
        "currency",
        F.upper(
            F.trim(
                F.col("currency")
            )
        )
    )

    # Standardisation statut
    .withColumn(
        "order_status",
        F.upper(
            F.trim(
                F.col("order_status")
            )
        )
    )
)

# ============================================================
# 4. DÉDUPLICATION MÉTIER
# ============================================================

dedup_window = (
    Window
    .partitionBy("order_id")
    .orderBy(
        F.col("updated_at").desc_nulls_last(),
        F.col("ingestion_timestamp").desc_nulls_last()
    )
)

orders_ranked = (
    normalized_orders
    .withColumn(
        "_row_number",
        F.row_number().over(dedup_window)
    )
)

duplicate_records = (
    orders_ranked
    .filter(
        F.col("_row_number") > 1
    )
)

deduplicated_orders = (
    orders_ranked
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

# ============================================================
# 5. RÈGLES DATA QUALITY
# ============================================================

allowed_statuses = [
    "CONFIRMED",
    "PROCESSING"
]

validated_orders = (
    deduplicated_orders

    .withColumn(
        "dq_order_id_null",
        F.col("order_id").isNull()
        | (F.col("order_id") == "")
    )

    .withColumn(
        "dq_customer_id_null",
        F.col("customer_id").isNull()
        | (F.col("customer_id") == "")
    )

    .withColumn(
        "dq_order_time_null",
        F.col("order_time").isNull()
    )

    .withColumn(
        "dq_invalid_amount",
        F.col("total_amount").isNull()
        | (F.col("total_amount") <= 0)
    )

    .withColumn(
        "dq_invalid_currency",
        F.col("currency").isNull()
        | (F.col("currency") != "MAD")
    )

    .withColumn(
        "dq_invalid_status",
        F.col("order_status").isNull()
        | (~F.col("order_status").isin(allowed_statuses))
    )
)

# ============================================================
# 6. CONSTRUIRE LA RAISON DU REJET
# ============================================================

validated_orders = (
    validated_orders

    .withColumn(
        "dq_errors",
        F.array_compact(
            F.array(

                F.when(
                    F.col("dq_order_id_null"),
                    F.lit("ORDER_ID_NULL")
                ),

                F.when(
                    F.col("dq_customer_id_null"),
                    F.lit("CUSTOMER_ID_NULL")
                ),

                F.when(
                    F.col("dq_order_time_null"),
                    F.lit("ORDER_TIME_NULL")
                ),

                F.when(
                    F.col("dq_invalid_amount"),
                    F.lit("INVALID_TOTAL_AMOUNT")
                ),

                F.when(
                    F.col("dq_invalid_currency"),
                    F.lit("INVALID_CURRENCY")
                ),

                F.when(
                    F.col("dq_invalid_status"),
                    F.lit("INVALID_ORDER_STATUS")
                )
            )
        )
    )

    .withColumn(
        "dq_is_valid",
        F.size(
            F.col("dq_errors")
        ) == 0
    )
)

# ============================================================
# 7. SÉPARER VALIDES ET INVALIDES
# ============================================================

valid_orders = (
    validated_orders
    .filter(
        F.col("dq_is_valid") == True
    )
)

invalid_orders = (
    validated_orders
    .filter(
        F.col("dq_is_valid") == False
    )
)

# ============================================================
# 8. SILVER : SUPPRIMER LES COLONNES DQ TECHNIQUES
# ============================================================

silver_orders = (
    valid_orders
    .drop(
        "dq_order_id_null",
        "dq_customer_id_null",
        "dq_order_time_null",
        "dq_invalid_amount",
        "dq_invalid_currency",
        "dq_invalid_status",
        "dq_errors",
        "dq_is_valid"
    )
)

# ============================================================
# 9. ÉCRITURE SILVER DELTA
# ============================================================

(
    silver_orders
    .write
    .format("delta")
    .mode("overwrite")
    .save(silver_orders_path)
)

# ============================================================
# 10. ÉCRITURE DES RECORDS INVALIDES
# ============================================================

(
    invalid_orders
    .write
    .format("delta")
    .mode("overwrite")
    .save(dq_invalid_orders_path)
)

# ============================================================
# 11. MÉTRIQUES DATA QUALITY
# ============================================================

records_checked = deduplicated_orders.count()

invalid_records = invalid_orders.count()

valid_records = silver_orders.count()

duplicate_count = duplicate_records.count()

if records_checked > 0:
    dq_pass_rate = (
        valid_records
        / records_checked
    ) * 100
else:
    dq_pass_rate = 0.0

dq_metrics = [
    (
        "orders",
        records_checked,
        valid_records,
        invalid_records,
        duplicate_count,
        float(dq_pass_rate)
    )
]

dq_metrics_schema = """
dataset STRING,
records_checked LONG,
valid_records LONG,
invalid_records LONG,
duplicate_records LONG,
dq_pass_rate DOUBLE
"""

dq_metrics_df = (
    spark.createDataFrame(
        dq_metrics,
        schema=dq_metrics_schema
    )
    .withColumn(
        "created_at",
        F.current_timestamp()
    )
)

(
    dq_metrics_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(dq_metrics_orders_path)
)

# ============================================================
# 12. AFFICHAGE DES RÉSULTATS
# ============================================================

print(
    "Bronze records :",
    bronze_orders.count()
)

print(
    "Records après déduplication :",
    records_checked
)

print(
    "Doublons détectés :",
    duplicate_count
)

print(
    "Records valides :",
    valid_records
)

print(
    "Records invalides :",
    invalid_records
)

print(
    "DQ pass rate :",
    round(dq_pass_rate, 2),
    "%"
)

print("\n=== SILVER ORDERS ===")

display(
    silver_orders
    .orderBy("order_id")
)

print("\n=== RECORDS INVALIDES ===")

display(
    invalid_orders.select(
        "order_id",
        "customer_id",
        "total_amount",
        "currency",
        "order_status",
        "dq_errors",
        "source_file",
        "batch_id"
    )
)

print("\n=== MÉTRIQUES DATA QUALITY ===")

display(
    dq_metrics_df
)